In [ ]:
# init
# One-off repair for this measurement: the breaking script selected a new
# current-amplifier gain before individual sweeps. get_traces() loaded the
# data with AmpI=1 above; now divide each current trace by its recorded gain.
# If several gains were tested, the last test before the actual sweep is the
# selected one (e.g. ampB=100 reproduces the following sweep, not ampB=10).
from pathlib import Path
import re
import h5py
import matplotlib.pyplot as plt
import superconductivity as sc


def reconstruct_amp_i(traces, filespec):
    h5path = Path(filespec.location) / filespec.h5path
    gains = []
    repaired = []

    with h5py.File(h5path, "r") as file:
        measurement = file["measurement"][filespec.measurement]
        for skey, trace in zip(traces.skeys, traces):
            candidates = []
            for key, group in measurement[skey].items():
                match = re.fullmatch(r"ampB=(\d+(?:\.\d+)?)", key)
                if match is None:
                    continue
                start = group["sweep"]["adwin"]["time"][0]
                candidates.append((float(start), float(match.group(1))))

            if not candidates:
                raise ValueError(f"No ampB gain recorded for {skey}")

            gain = max(candidates)[1]
            gains.append(gain)
            temperature = None if trace.Tsample_K is None else trace.Tsample_K.value
            repaired.append(
                sceva.Trace(
                    I_nA=trace.I_nA.values / gain,
                    V_mV=trace.V_mV.values,
                    t_s=trace.t_s.values,
                    Tsample_K=temperature,
                )
            )

    repaired_traces = sceva.Traces(
        traces=repaired,
        skeys=traces.skeys,
        indices=traces.indices,
        yaxis=traces.yaxis,
    )
    return repaired_traces, np.asarray(gains)


def get_amplitude(power_dbm, R=50):
    """
    Convert power in dBm to real voltage amplitude (peak).

    Parameters
    ----------
    power_dbm : float or np.ndarray
        Power in dBm.
    R : float, optional
        System impedance in ohms. Defaults to 50 ohms.

    Returns
    -------
    amplitude : float or np.ndarray
        Corresponding voltage amplitude (in volts).

    Notes
    -----
    Inverts the formula:
        power = V^2 / R,
    after converting dBm to watts.
    """
    power_watts = 10 ** (power_dbm / 10) * 1e-3  # Convert dBm to watts
    amplitude = np.sqrt(power_watts * R)  # Solve for voltage
    return amplitude

In [ ]:
# breaking (atomic contact)
import numpy as np
import superconductivity.evaluation as sceva
import superconductivity.utilities as scutil

cachepath = "breaking/"

# # make cache
# cache = scutil.make_cache(name="cache", path=cachepath)
# scutil.save_cache(cache)

# load cache
cache = scutil.load_cache("cache", path=cachepath)

# specs
cache.filespec = sceva.FileSpec(
    h5path="23_12_08_S22_breaking_4.hdf5",
    location="/Volumes/speedyboy/atomic-contact/breaking",
    measurement="positions",
)
cache.keysspec = sceva.KeysSpec(
    strip0="pos=+",
    strip1=None,
    # limits=(3400, 3788), # 60 G0
    limits=(3520, 3788),  # 4 G0
    # limits=(3539, 3788),  # 2 G0
    norm=1,
    label="x_arbu",
)

cache.tracespec_up = sceva.TraceSpec(
    AmpV=1000,
    AmpI=1,
    Rref_Ohm=51689,
    trigger_values=1,
    skip_edges=5,
)
cache.tracespec_down = sceva.TraceSpec(
    AmpV=1000,
    AmpI=1,
    Rref_Ohm=51689,
    trigger_values=2,
    skip_edges=5,
)
cache.samplingspec = sceva.SamplingSpec(
    Vbins_mV=np.linspace(-0.9, 0.9, 1801),
    Ibins_nA=np.linspace(-50, 50, 2001),
    cutoff_Hz=13.7,
    sampling_Hz=1370.0,
    median_bins=3,
    sigma_bins=2.0,
)
_ = scutil.save_cache(cache)

# get keys
mkeys = cache.filespec.mkeys()
skeys = cache.filespec.skeys()
cache.skeys = sceva.get_keys(
    filespec=cache.filespec,
    keysspec=cache.keysspec,
)
_ = scutil.save_cache(cache)

# get traces
cache.traces_up = sceva.get_traces(
    filespec=cache.filespec,
    keysspec=cache.keysspec,
    tracespec=cache.tracespec_up,
)
cache.traces_down = sceva.get_traces(
    filespec=cache.filespec,
    keysspec=cache.keysspec,
    tracespec=cache.tracespec_down,
)

cache.traces_up, cache.AmpI = reconstruct_amp_i(cache.traces_up, cache.filespec)
cache.traces_down, cache.AmpI = reconstruct_amp_i(cache.traces_down, cache.filespec)
_ = scutil.save_cache(cache)

# # offset analysis
# cache.offsetspec = sceva.OffsetSpec(
#     Vbins_mV=np.linspace(-0.5, 0.5, 51),
#     Ibins_nA=np.linspace(-3.0, 3.0, 51),
#     Voffscan_mV=np.linspace(-0.045, 0.045, 451),
#     Ioffscan_nA=np.linspace(-0.35, 0.35, 351),
#     cutoff_Hz=13.7,
#     sampling_Hz=1370.0,
# )
# cache.offsetanalysis = sceva.offset_analysis(
#     traces=cache.traces,
#     spec=cache.offsetspec,
# )
# cache.samplingspec.Voff_mV = cache.offsetanalysis.Voff_mV
# cache.samplingspec.Ioff_nA = cache.offsetanalysis.Ioff_nA
# _ = scutil.save_cache(cache)

# sampling
cache.exp_v_up, cache.exp_i_up = sceva.sample(
    traces=cache.traces_up,
    samplingspec=cache.samplingspec,
)
cache.exp_v_down, cache.exp_i_down = sceva.sample(
    traces=cache.traces_down,
    samplingspec=cache.samplingspec,
)
_ = scutil.save_cache(cache)

# save data
np.savez_compressed(
    f"{cachepath}/eva.npz",
    Vbias_mV=cache.exp_v_up["V_mV"],
    x_arbu=cache.exp_v_up["x_arbu"],
    Iup_nA=cache.exp_v_up["I_nA"],
    dGup_G0=cache.exp_v_up["dG_G0"],
    Idown_nA=cache.exp_v_down["I_nA"],
    dGdown_G0=cache.exp_v_down["dG_G0"],
    Tbath_K=cache.traces_up.Tsample_K,
)

In [ ]:
# breaking (tunneling)
import numpy as np
import superconductivity.evaluation as sceva
import superconductivity.utilities as scutil

cachepath = "breaking/"

# # make cache
# cache = scutil.make_cache(name="cache", path=cachepath)
# scutil.save_cache(cache)

# load cache
cache = scutil.load_cache("cache", path=cachepath)

# specs
cache.filespec = sceva.FileSpec(
    h5path="23_12_08_S22_breaking_4.hdf5",
    location="/Volumes/speedyboy/atomic-contact/breaking",
    measurement="positions",
)
cache.keysspec = sceva.KeysSpec(
    strip0="pos=+",
    strip1=None,
    # limits=(3400, 3788), # 60 G0
    limits=(3782, 3932),  # 4 G0
    # limits=(3539, 3788),  # 2 G0
    norm=1,
    label="x_arbu",
)

cache.tracespec_up = sceva.TraceSpec(
    AmpV=1000,
    AmpI=1,
    Rref_Ohm=51689,
    trigger_values=1,
    skip_edges=5,
)
cache.tracespec_down = sceva.TraceSpec(
    AmpV=1000,
    AmpI=1,
    Rref_Ohm=51689,
    trigger_values=2,
    skip_edges=5,
)
cache.samplingspec = sceva.SamplingSpec(
    Vbins_mV=np.linspace(-0.9, 0.9, 1801),
    Ibins_nA=np.linspace(-50, 50, 2001),
    cutoff_Hz=13.7,
    sampling_Hz=1370.0,
    median_bins=3,
    sigma_bins=2.0,
)
_ = scutil.save_cache(cache)

# get keys
mkeys = cache.filespec.mkeys()
skeys = cache.filespec.skeys()
cache.skeys = sceva.get_keys(
    filespec=cache.filespec,
    keysspec=cache.keysspec,
)
_ = scutil.save_cache(cache)

# get traces
cache.traces_up = sceva.get_traces(
    filespec=cache.filespec,
    keysspec=cache.keysspec,
    tracespec=cache.tracespec_up,
)
cache.traces_down = sceva.get_traces(
    filespec=cache.filespec,
    keysspec=cache.keysspec,
    tracespec=cache.tracespec_down,
)

cache.traces_up, cache.AmpI = reconstruct_amp_i(cache.traces_up, cache.filespec)
cache.traces_down, cache.AmpI = reconstruct_amp_i(cache.traces_down, cache.filespec)
_ = scutil.save_cache(cache)

# # offset analysis
# cache.offsetspec = sceva.OffsetSpec(
#     Vbins_mV=np.linspace(-0.5, 0.5, 51),
#     Ibins_nA=np.linspace(-3.0, 3.0, 51),
#     Voffscan_mV=np.linspace(-0.045, 0.045, 451),
#     Ioffscan_nA=np.linspace(-0.35, 0.35, 351),
#     cutoff_Hz=13.7,
#     sampling_Hz=1370.0,
# )
# cache.offsetanalysis = sceva.offset_analysis(
#     traces=cache.traces,
#     spec=cache.offsetspec,
# )
# cache.samplingspec.Voff_mV = cache.offsetanalysis.Voff_mV
# cache.samplingspec.Ioff_nA = cache.offsetanalysis.Ioff_nA
# _ = scutil.save_cache(cache)

# sampling
cache.exp_v_up, cache.exp_i_up = sceva.sample(
    traces=cache.traces_up,
    samplingspec=cache.samplingspec,
)
cache.exp_v_down, cache.exp_i_down = sceva.sample(
    traces=cache.traces_down,
    samplingspec=cache.samplingspec,
)
_ = scutil.save_cache(cache)

# save data
np.savez_compressed(
    f"{cachepath}/tunnel_eva.npz",
    Vbias_mV=cache.exp_v_up["V_mV"],
    x_arbu=cache.exp_v_up["x_arbu"],
    Iup_nA=cache.exp_v_up["I_nA"],
    dGup_G0=cache.exp_v_up["dG_G0"],
    Idown_nA=cache.exp_v_down["I_nA"],
    dGdown_G0=cache.exp_v_down["dG_G0"],
    Tbath_K=cache.traces_up.Tsample_K,
)

In [ ]:
# 4.3 G0
import numpy as np
import superconductivity.evaluation as sceva
import superconductivity.utilities as scutil

for i, nui_GHz in enumerate([7.8, 15.0, 19.3]):
    cachepath = f"4.3G0/{nui_GHz:.1f}GHz_stripline/"

    # # make cache
    # cache = scutil.make_cache(name="cache", path=cachepath)
    # scutil.save_cache(cache)

    # load cache
    cache = scutil.load_cache("cache", path=cachepath)

    # specs
    cache.filespec = sceva.FileSpec(
        h5path="23_11_15_PR22e9_4.3G_1.hdf5",
        location="/Volumes/speedyboy/atomic-contact/4.30 G_0/",
        measurement=f"vna powers_{nui_GHz:.1f}00GHz_+0.00mT",
    )
    cache.keysspec = sceva.KeysSpec(
        strip0="P=",
        strip1="dBm",
        remove_key="no_power",
        add_key=[
            ("no_power", -1e9),
        ],
        limits=(None, None),
        norm=1,
        label="Pout_dBm",
    )
    cache.tracespec = sceva.TraceSpec(
        AmpV=1000,
        AmpI=100,
        Rref_Ohm=51689,
        trigger_values=1,
        skip_edges=5,
    )
    cache.offsetspec = sceva.OffsetSpec(
        Vbins_mV=np.linspace(-0.5, 0.5, 51),
        Ibins_nA=np.linspace(-300.0, 300.0, 51),
        Voffscan_mV=np.linspace(-0.045, 0.045, 451),
        Ioffscan_nA=np.linspace(-35, 35, 351),
        cutoff_Hz=13.7,
        sampling_Hz=137.0,
    )
    cache.samplingspec = sceva.SamplingSpec(
        Vbins_mV=np.linspace(-1.0, 1.0, 1001),
        Ibins_nA=np.linspace(-600, 600, 1001),
        cutoff_Hz=13.7,
        sampling_Hz=1370.0,
        median_bins=3,
        sigma_bins=2.0,
    )
    _ = scutil.save_cache(cache)

    # get keys
    mkeys = cache.filespec.mkeys()
    skeys = cache.filespec.skeys()
    cache.skeys = sceva.get_keys(
        filespec=cache.filespec,
        keysspec=cache.keysspec,
    )
    _ = scutil.save_cache(cache)

    # get traces
    cache.traces = sceva.get_traces(
        filespec=cache.filespec,
        keysspec=cache.keysspec,
        tracespec=cache.tracespec,
    )
    _ = scutil.save_cache(cache)

    # offset analysis
    # cache.offsetanalysis = sceva.offset_analysis(
    #     traces=cache.traces,
    #     spec=cache.offsetspec,
    # )
    cache.samplingspec.Voff_mV = cache.offsetanalysis.Voff_mV
    cache.samplingspec.Ioff_nA = cache.offsetanalysis.Ioff_nA
    _ = scutil.save_cache(cache)

    # sampling
    cache.exp_v, cache.exp_i = sceva.sample(
        traces=cache.traces,
        samplingspec=cache.samplingspec,
    )
    _ = scutil.save_cache(cache)

    # save data
    np.savez_compressed(
        f"{cachepath}/eva.npz",
        Vbias_mV=cache.exp_v["V_mV"],
        Ibias_nA=cache.exp_i["I_nA"],
        Pout_dBm=cache.exp_v["Pout_dBm"],
        Aout_mV=get_amplitude(cache.exp_v["Pout_dBm"]),
        nu_GHz=nui_GHz,
        dGexp_G0=cache.exp_v["dG_G0"],
        dRexp_R0=cache.exp_i["dR_R0"],
        Iexp_nA=cache.exp_v["I_nA"],
        Tbath_K=cache.traces.Tsample_K,
    )

In [ ]:
# 43.0 G0
import numpy as np
import superconductivity.evaluation as sceva
import superconductivity.utilities as scutil

for coupling in ["antenna", "stripline"]:

    for i, nui_GHz in enumerate([7.8, 15.0, 19.3]):
        cachepath = f"43.0G0/{nui_GHz:.1f}GHz_{coupling}/"

        # # make cache
        # cache = scutil.make_cache(name="cache", path=cachepath)
        # scutil.save_cache(cache)

        # load cache
        cache = scutil.load_cache("cache", path=cachepath)

        # specs
        if nui_GHz == 15.0:
            cache.filespec = sceva.FileSpec(
                h5path=f"2023-10-20_MWIV_{coupling}_powers_25_G0_0.hdf5",
                location="/Volumes/speedyboy/atomic-contact/43.00 G_0/",
                measurement=f"frequency_at_15GHz",
            )
        else:
            cache.filespec = sceva.FileSpec(
                h5path=f"2023-10-20_MWIV_{coupling}_powers_frequencies_25_G0_0.hdf5",
                location="/Volumes/speedyboy/atomic-contact/43.00 G_0/",
                measurement=f"frequency_at_{nui_GHz}GHz",
            )

        cache.keysspec = sceva.KeysSpec(
            strip0="nu=",
            strip1="dBm",
            remove_key="nu=-31.0dBm",
            add_key=[
                ("nu=-31.0dBm", -1e9),
            ],
            limits=(None, None),
            norm=1,
            label="Pout_dBm",
        )
        cache.tracespec = sceva.TraceSpec(
            AmpV=10_000,
            AmpI=100,
            Rref_Ohm=51689,
            trigger_values=1,
            skip_edges=5,
        )
        cache.offsetspec = sceva.OffsetSpec(
            Vbins_mV=np.linspace(-0.5, 0.5, 101),
            Ibins_nA=np.linspace(-1700.0, 1700.0, 51),
            Voffscan_mV=np.linspace(-0.045, 0.045, 451),
            Ioffscan_nA=np.linspace(-70, 70, 351),
            cutoff_Hz=13.7,
            sampling_Hz=137.0,
        )
        cache.samplingspec = sceva.SamplingSpec(
            Vbins_mV=np.linspace(-0.5, 0.5, 1001),
            Ibins_nA=np.linspace(-3400, 3400, 1001),
            cutoff_Hz=13.7,
            sampling_Hz=1370.0,
            median_bins=3,
            sigma_bins=2.0,
        )
        # _ = scutil.save_cache(cache)

        # get keys
        mkeys = cache.filespec.mkeys()
        skeys = cache.filespec.skeys()
        cache.skeys = sceva.get_keys(
            filespec=cache.filespec,
            keysspec=cache.keysspec,
        )
        # _ = scutil.save_cache(cache)

        # get traces
        cache.traces = sceva.get_traces(
            filespec=cache.filespec,
            keysspec=cache.keysspec,
            tracespec=cache.tracespec,
        )
        _ = scutil.save_cache(cache)

        # offset analysis
        # cache.offsetanalysis = sceva.offset_analysis(
        #     traces=cache.traces,
        #     spec=cache.offsetspec,
        # )
        cache.samplingspec.Voff_mV = cache.offsetanalysis.Voff_mV
        cache.samplingspec.Ioff_nA = cache.offsetanalysis.Ioff_nA
        _ = scutil.save_cache(cache)

        # sampling
        cache.exp_v, cache.exp_i = sceva.sample(
            traces=cache.traces,
            samplingspec=cache.samplingspec,
        )
        _ = scutil.save_cache(cache)

        # save data
        np.savez_compressed(
            f"{cachepath}/eva.npz",
            Vbias_mV=cache.exp_v["V_mV"],
            Ibias_nA=cache.exp_i["I_nA"],
            Pout_dBm=cache.exp_v["Pout_dBm"],
            Aout_mV=get_amplitude(cache.exp_v["Pout_dBm"]),
            nu_GHz=nui_GHz,
            dGexp_G0=cache.exp_v["dG_G0"],
            dRexp_R0=cache.exp_i["dR_R0"],
            Iexp_nA=cache.exp_v["I_nA"],
            Tbath_K=cache.traces.Tsample_K,
        )

In [ ]:
# 550 G0, magnetic field
import numpy as np
import superconductivity.evaluation as sceva
import superconductivity.utilities as scutil

cachepath = f"550G0/magnetic-field/"

# # make cache
# cache = scutil.make_cache(name="cache", path=cachepath)
# scutil.save_cache(cache)

# load cache
cache = scutil.load_cache("cache", path=cachepath)

# specs
cache.filespec = sceva.FileSpec(
    h5path="2023-08-29_HIV_23.5Ohm_3_ADwin.hdf5",
    location="/Volumes/speedyboy/atomic-contact/unbroken",
    measurement="critical field",
)
cache.keysspec = sceva.KeysSpec(
    strip0="uH=",
    strip1="mT",
    norm=1,
    limits=(60, 171),
    label="uH_mT",
)
cache.tracespec = sceva.TraceSpec(
    AmpV=100,
    AmpI=10,
    Rref_Ohm=100,
    trigger_values=1,
    skip_edges=5,
)
cache.samplingspec = sceva.SamplingSpec(
    Vbins_mV=np.linspace(-18, 28, 921),
    Ibins_nA=np.linspace(-0.7e6, 1.2e6, 951),
    cutoff_Hz=103.7,
    sampling_Hz=1370.0,
    median_bins=3,
    sigma_bins=2.0,
)
_ = scutil.save_cache(cache)

# get keys
mkeys = cache.filespec.mkeys()
skeys = cache.filespec.skeys()
cache.skeys = sceva.get_keys(
    filespec=cache.filespec,
    keysspec=cache.keysspec,
)
_ = scutil.save_cache(cache)

# # get traces
# cache.traces = sceva.get_traces(
#     filespec=cache.filespec,
#     keysspec=cache.keysspec,
#     tracespec=cache.tracespec,
# )
# _ = scutil.save_cache(cache)

# sampling
cache.exp_v, cache.exp_i = sceva.sample(
    traces=cache.traces,
    samplingspec=cache.samplingspec,
)
_ = scutil.save_cache(cache)

# save data
np.savez_compressed(
    f"{cachepath}/eva.npz",
    Vbias_mV=cache.exp_v["V_mV"],
    Ibias_nA=cache.exp_i["I_nA"],
    uH_mT=cache.exp_v["uH_mT"],
    dGexp_G0=cache.exp_v["dG_G0"],
    dRexp_R0=cache.exp_i["dR_R0"],
    Iexp_nA=cache.exp_v["I_nA"],
    Vexp_mV=cache.exp_i["V_mV"],
    Tbath_K=cache.traces.Tsample_K,
)

In [ ]:
# 0.05 G0, 15GHz, stripline
import numpy as np
import superconductivity.evaluation as sceva
import superconductivity.utilities as scutil

cachepath = f"0.05G0/15.0GHz_stripline/"

# # make cache
# cache = scutil.make_cache(name="cache", path=cachepath)
# scutil.save_cache(cache)

# load cache
cache = scutil.load_cache("cache", path=cachepath)

# specs
cache.filespec = sceva.FileSpec(
    h5path="2023-10-27_HIV_tunnel_contact_1.hdf5",
    location="/Volumes/speedyboy/atomic-contact/0.05 G_0",
    measurement="frequency_at_15GHz",
)
cache.keysspec = sceva.KeysSpec(
    strip0="nu=",
    strip1="dBm",
    remove_key="nu=-31.0dBm",
    add_key=[
        ("nu=-31.0dBm", -1e9),
    ],
    limits=(None, None),
    norm=1,
    label="Pout_dBm",
)
cache.tracespec = sceva.TraceSpec(
    AmpV=1000,
    AmpI=10000,
    Rref_Ohm=51689,
    trigger_values=1,
    skip_edges=5,
)
cache.offsetspec = sceva.OffsetSpec(
    Vbins_mV=np.linspace(-0.5, 0.5, 51),
    Ibins_nA=np.linspace(-3.0, 3.0, 51),
    Voffscan_mV=np.linspace(-0.005, 0.005, 501),
    Ioffscan_nA=np.linspace(-0.03, 0.03, 501),
    cutoff_Hz=7.3,
    sampling_Hz=1370.0,
    median_bins=3,
    sigma_bins=2.0,
)
cache.samplingspec = sceva.SamplingSpec(
    Vbins_mV=np.linspace(-1.6, 1.6, 2001),
    Ibins_nA=np.linspace(-6, 6, 1801),
    cutoff_Hz=7.3,
    sampling_Hz=1370.0,
    median_bins=3,
    sigma_bins=2.0,
)
_ = scutil.save_cache(cache)

# get keys
mkeys = cache.filespec.mkeys()
skeys = cache.filespec.skeys()
cache.skeys = sceva.get_keys(
    filespec=cache.filespec,
    keysspec=cache.keysspec,
)
_ = scutil.save_cache(cache)

# get traces
cache.traces = sceva.get_traces(
    filespec=cache.filespec,
    keysspec=cache.keysspec,
    tracespec=cache.tracespec,
)
_ = scutil.save_cache(cache)

# offset analysis
# cache.offsetanalysis = sceva.offset_analysis(
#     traces=cache.traces,
#     spec=cache.offsetspec,
# )
cache.samplingspec.Voff_mV = cache.offsetanalysis.Voff_mV
cache.samplingspec.Ioff_nA = cache.offsetanalysis.Ioff_nA
_ = scutil.save_cache(cache)

# sampling
cache.exp_v, cache.exp_i = sceva.sample(
    traces=cache.traces,
    samplingspec=cache.samplingspec,
)
_ = scutil.save_cache(cache)

Aout_mV = 1e3 * get_amplitude(cache.exp_v["Pout_dBm"])

# save data
np.savez_compressed(
    f"{cachepath}/eva.npz",
    Vbias_mV=cache.exp_v["V_mV"],
    Ibias_nA=cache.exp_i["I_nA"],
    Pout_dBm=cache.exp_v["Pout_dBm"],
    Aout_mV=Aout_mV,
    nu_GHz=15.0,
    dGexp_G0=cache.exp_v["dG_G0"],
    dRexp_R0=cache.exp_i["dR_R0"],
    Iexp_nA=cache.exp_v["I_nA"],
    Tbath_K=cache.traces.Tsample_K,
)

# calibrate
cache.calibrationspec = sceva.CalibrationSpec(
    label="Aout_mV",
    lookup=Aout_mV,
)
cache.cal_v, cache.cal_i = sceva.calibrate(
    exp_v=cache.exp_v,
    exp_i=cache.exp_i,
    calibrationspec=cache.calibrationspec,
)
_ = scutil.save_cache(cache)

Abin_mV = np.linspace(0, 251, 101)
Tsample_K = np.array(cache.traces.Tsample_K)
Tbath_K = sc.bin_y_over_x(Tsample_K, Aout_mV, Abin_mV)

cache.mapped_v, cache.mapped_i = scutil.mapping(
    (cache.cal_v, cache.cal_i),
    axis="Aout_mV",
    # N_up=10,
    xbins=Abin_mV,
    fill="nearest",
)
_ = scutil.save_cache(cache)

np.savez_compressed(
    f"{cachepath}/cal.npz",
    Vbias_mV=cache.mapped_v["V_mV"],
    Ibias_nA=cache.mapped_i["I_nA"],
    Aout_mV=cache.mapped_v["Aout_mV"],
    nu_GHz=15.0,
    dGexp_G0=cache.mapped_v["dG_G0"],
    dRexp_R0=cache.mapped_i["dR_R0"],
    Iexp_nA=cache.mapped_v["I_nA"],
    Tbath_K=Tbath_K,
)

In [ ]:
# import matplotlib.pyplot as plt

# plt.imshow(
#     # cache.exp_v["dG_G0"],
#     # cache.exp_i["dR_R0"],
#     cache.offsetanalysis["dRerr_R0"],
#     # cache.offsetanalysis["dGerr_G0"],
#     aspect="auto",
#     clim=(0, 1),
#     interpolation="None",
# )

In [ ]:
# 0.05 G0, magnetic field
import numpy as np
import superconductivity.evaluation as sceva
import superconductivity.utilities as scutil

cachepath = f"0.05G0/magnetic-field/"

# # make cache
# cache = scutil.make_cache(name="cache", path=cachepath)
# scutil.save_cache(cache)

# load cache
cache = scutil.load_cache("cache", path=cachepath)

# specs
cache.filespec = sceva.FileSpec(
    h5path="2023-10-27_HIV_tunnel_contact_1.hdf5",
    location="/Volumes/speedyboy/atomic-contact/0.05 G_0",
    measurement="critical field",
)
cache.keysspec = sceva.KeysSpec(
    strip0="uH=",
    strip1="mT",
    norm=1,
    label="uH_mT",
)
cache.tracespec = sceva.TraceSpec(
    AmpV=1000,
    AmpI=10000,
    Rref_Ohm=51689,
    trigger_values=1,
    skip_edges=5,
)
cache.samplingspec = sceva.SamplingSpec(
    Vbins_mV=np.linspace(-0.6, 0.6, 1001),
    Ibins_nA=np.linspace(-2, 2, 1001),
    cutoff_Hz=7.3,
    sampling_Hz=1370.0,
    median_bins=9,
    sigma_bins=3.0,
)
_ = scutil.save_cache(cache)

# get keys
mkeys = cache.filespec.mkeys()
skeys = cache.filespec.skeys()
cache.skeys = sceva.get_keys(
    filespec=cache.filespec,
    keysspec=cache.keysspec,
)
_ = scutil.save_cache(cache)

# # get traces
# cache.traces = sceva.get_traces(
#     filespec=cache.filespec,
#     keysspec=cache.keysspec,
#     tracespec=cache.tracespec,
# )
# _ = scutil.save_cache(cache)

# sampling
cache.exp_v, cache.exp_i = sceva.sample(
    traces=cache.traces,
    samplingspec=cache.samplingspec,
)
_ = scutil.save_cache(cache)

# save data
np.savez_compressed(
    f"{cachepath}/eva.npz",
    Vbias_mV=cache.exp_v["V_mV"],
    Ibias_nA=cache.exp_i["I_nA"],
    uH_mT=cache.exp_v["uH_mT"],
    dGexp_G0=cache.exp_v["dG_G0"],
    dRexp_R0=cache.exp_i["dR_R0"],
    Iexp_nA=cache.exp_v["I_nA"],
    Tbath_K=cache.traces.Tsample_K,
)

In [ ]:
# 0.05 G0, temperature
import numpy as np
import superconductivity.evaluation as sceva
import superconductivity.utilities as scutil
from superconductivity.utilities.transport import mapping

cachepath = f"0.05G0/temperature/"

# # make cache
# cache = scutil.make_cache(name="cache", path=cachepath)
# scutil.save_cache(cache)

# load cache
cache = scutil.load_cache("cache", path=cachepath)

# specs
cache.filespec = sceva.FileSpec(
    h5path="23_12_22_time_evolution_over_warm_up_0.hdf5",
    location="/Volumes/speedyboy/atomic-contact/warm up",
    measurement="time evolution",
)
cache.keysspec = sceva.KeysSpec(
    strip0="i==",
    limits=[(None, 325), (1050, 1250), (3900, 4700)],
    norm=1,
    label="i",
)
cache.tracespec = sceva.TraceSpec(
    AmpV=1000,
    AmpI=1000,
    Rref_Ohm=51689,
    trigger_values=1,
    skip_edges=5,
)
cache.samplingspec = sceva.SamplingSpec(
    Vbins_mV=np.linspace(-0.6, 0.6, 1001),
    Ibins_nA=np.linspace(-6, 6, 1001),
    cutoff_Hz=13.7,
    sampling_Hz=1370.0,
    median_bins=3,
    sigma_bins=2.0,
)
_ = scutil.save_cache(cache)

# get keys
mkeys = cache.filespec.mkeys()
skeys = cache.filespec.skeys()
cache.skeys = sceva.get_keys(
    filespec=cache.filespec,
    keysspec=cache.keysspec,
)
_ = scutil.save_cache(cache)

# # get traces
# cache.traces = sceva.get_traces(
#     filespec=cache.filespec,
#     keysspec=cache.keysspec,
#     tracespec=cache.tracespec,
# )
# _ = scutil.save_cache(cache)

# sampling
cache.exp_v, cache.exp_i = sceva.sample(
    traces=cache.traces,
    samplingspec=cache.samplingspec,
)
_ = scutil.save_cache(cache)

# calibrate
cache.calibrationspec = sceva.CalibrationSpec(
    label="Tbath_K",
    lookup=cache.traces.Tsample_K,
)
cache.cal_v, cache.cal_i = sceva.calibrate(
    exp_v=cache.exp_v,
    exp_i=cache.exp_i,
    calibrationspec=cache.calibrationspec,
)
_ = scutil.save_cache(cache)

cache.mapped_v, cache.mapped_i = mapping(
    (cache.cal_v, cache.cal_i),
    axis="Tbath_K",
    N_up=1,
    xbins=np.linspace(0, 1.31, 132),
    fill=None,
)
_ = scutil.save_cache(cache)

# save data
np.savez_compressed(
    f"{cachepath}/eva.npz",
    Vbias_mV=cache.mapped_v["V_mV"],
    Ibias_nA=cache.mapped_i["I_nA"],
    Tbath_K=cache.mapped_v["Tbath_K"],
    dGexp_G0=cache.mapped_v["dG_G0"],
    dRexp_R0=cache.mapped_i["dR_R0"],
    Iexp_nA=cache.mapped_v["I_nA"],
)